# Yonder ML take-home: mallet / bottle detection (GPU run)

Reproduces the whole pipeline on a Colab GPU in roughly 15 minutes: download, audit, grouped split, CLAHE, training
(baseline, final, leak comparison), evaluation, inference. **Runtime > Change runtime type > GPU** first.

Your Roboflow API key is read with `getpass` (not echoed, not saved in the notebook) and is only kept in this
session's environment.

In [ ]:
!git clone https://github.com/Ryan-Chow/yonder-ml-take-home.git
%cd yonder-ml-take-home
!pip install -q -r requirements.txt
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
import os, getpass
if not os.environ.get("ROBOFLOW_API_KEY"):
    os.environ["ROBOFLOW_API_KEY"] = getpass.getpass("Roboflow API key: ")
!python scripts/download_data.py

In [ ]:
!python scripts/audit_data.py
!python scripts/make_split.py
!python -m src.preprocess --src data/split --dst data/split_clahe
!python -m src.preprocess --src data/orig --dst data/orig_clahe

Smoke test first (1 epoch, 20 images), then the 2x2 ablation (baseline, clahe_only, aug_only, final), the leakage run, the paired bootstrap, and `ship.py`, which picks the shipped arm by the pre-registered rule. `--device 0` = first CUDA GPU.

In [ ]:
!python scripts/train.py --run final --smoke --device 0
for run in ["baseline", "clahe_only", "aug_only", "final", "leak"]:
    !python scripts/train.py --run {run} --device 0
!python scripts/bootstrap_ablation.py > /dev/null
!python scripts/ship.py

In [ ]:
!python scripts/evaluate.py --skip-mps
from IPython.display import Markdown, Image, display
display(Markdown(open("results/metrics.md").read()))

In [ ]:
!python scripts/error_analysis.py > /dev/null
!python predict.py data/split/valid/images outputs/predictions --save-vis
!head -5 $(ls outputs/predictions/*.txt | head -1)
display(Image("results/errors_sheet.jpg"))

In [ ]:
!python -m pytest -q